In [26]:
import http.client
import os 
import json
from dotenv import load_dotenv
import pandas as pd
from sqlalchemy import create_engine

load_dotenv()

conn = http.client.HTTPSConnection("api.collectapi.com")

headers = { 
    'content-type': "application/json", 
    'authorization': f"apikey {os.getenv('GASPRICES_API_KEY')}" 
}

conn.request("GET", "/gasPrice/stateUsaPrice?state=WA", headers=headers)

res = conn.getresponse()
data = res.read()

gas_data = json.loads(data.decode("utf-8"))

print(json.dumps(gas_data, indent=4))


{
    "success": true,
    "result": {
        "state": [
            {
                "currency": "usd",
                "name": "Washington",
                "lowername": "washington",
                "gasoline": "5.5290",
                "midGrade": "5.8180",
                "premium": "6.0770",
                "diesel": "6.4810"
            }
        ],
        "cities": [
            {
                "currency": "usd",
                "gasoline": "5.4210",
                "midGrade": "5.5380",
                "premium": "5.9300",
                "diesel": "6.2280",
                "name": "Bellingham",
                "lowername": "bellingham"
            },
            {
                "currency": "usd",
                "gasoline": "5.7050",
                "midGrade": "5.9440",
                "premium": "6.1870",
                "diesel": "6.5010",
                "name": "Bremerton",
                "lowername": "bremerton"
            },
            {
                "curr

In [27]:
type(gas_data)

dict

In [28]:
type(gas_data)
gas_data

{'success': True,
 'result': {'state': [{'currency': 'usd',
    'name': 'Washington',
    'lowername': 'washington',
    'gasoline': '5.5290',
    'midGrade': '5.8180',
    'premium': '6.0770',
    'diesel': '6.4810'}],
  'cities': [{'currency': 'usd',
    'gasoline': '5.4210',
    'midGrade': '5.5380',
    'premium': '5.9300',
    'diesel': '6.2280',
    'name': 'Bellingham',
    'lowername': 'bellingham'},
   {'currency': 'usd',
    'gasoline': '5.7050',
    'midGrade': '5.9440',
    'premium': '6.1870',
    'diesel': '6.5010',
    'name': 'Bremerton',
    'lowername': 'bremerton'},
   {'currency': 'usd',
    'gasoline': '5.4970',
    'midGrade': '5.6880',
    'premium': '5.9380',
    'diesel': '6.6340',
    'name': 'Olympia',
    'lowername': 'olympia'},
   {'currency': 'usd',
    'gasoline': '5.1780',
    'midGrade': '5.4580',
    'premium': '5.7050',
    'diesel': '6.1190',
    'name': 'Richland-Kennewick-Pasco',
    'lowername': 'richland-kennewick-pasco'},
   {'currency': 'usd',

In [29]:
print(gas_data.keys())


dict_keys(['success', 'result'])


In [30]:
gasprices = [gas_data['result']]
type(gasprices)
gasprices


[{'state': [{'currency': 'usd',
    'name': 'Washington',
    'lowername': 'washington',
    'gasoline': '5.5290',
    'midGrade': '5.8180',
    'premium': '6.0770',
    'diesel': '6.4810'}],
  'cities': [{'currency': 'usd',
    'gasoline': '5.4210',
    'midGrade': '5.5380',
    'premium': '5.9300',
    'diesel': '6.2280',
    'name': 'Bellingham',
    'lowername': 'bellingham'},
   {'currency': 'usd',
    'gasoline': '5.7050',
    'midGrade': '5.9440',
    'premium': '6.1870',
    'diesel': '6.5010',
    'name': 'Bremerton',
    'lowername': 'bremerton'},
   {'currency': 'usd',
    'gasoline': '5.4970',
    'midGrade': '5.6880',
    'premium': '5.9380',
    'diesel': '6.6340',
    'name': 'Olympia',
    'lowername': 'olympia'},
   {'currency': 'usd',
    'gasoline': '5.1780',
    'midGrade': '5.4580',
    'premium': '5.7050',
    'diesel': '6.1190',
    'name': 'Richland-Kennewick-Pasco',
    'lowername': 'richland-kennewick-pasco'},
   {'currency': 'usd',
    'gasoline': '5.7980',
 

In [31]:
# Extract city level data
city_data = gas_data['result']['cities']
cities_df = pd.DataFrame(city_data)
cities_df.head()


,currency,gasoline,midGrade,premium,diesel,name,lowername
0,usd,5.4210,5.5380,5.9300,6.2280,Bellingham,bellingham
1,usd,5.7050,5.9440,6.1870,6.5010,Bremerton,bremerton
2,usd,5.4970,5.6880,5.9380,6.6340,Olympia,olympia
3,usd,5.1780,5.4580,5.7050,6.1190,Richland-Kennewick-Pasco,richland-kennewick-pasco
4,usd,5.7980,6.0620,6.3070,6.7640,Seattle-Bellevue-Everett,seattle-bellevue-everett


In [32]:
# drop the lowername column, rename name to cities

cities_df = cities_df.drop(columns=['lowername'])

cities_df = cities_df.rename(columns={'name': 'cities'})

cities_df.head()


,currency,gasoline,midGrade,premium,diesel,cities
0,usd,5.4210,5.5380,5.9300,6.2280,Bellingham
1,usd,5.7050,5.9440,6.1870,6.5010,Bremerton
2,usd,5.4970,5.6880,5.9380,6.6340,Olympia
3,usd,5.1780,5.4580,5.7050,6.1190,Richland-Kennewick-Pasco
4,usd,5.7980,6.0620,6.3070,6.7640,Seattle-Bellevue-Everett


In [33]:
# store to db

DB_HOST = os.getenv('DB_HOST')
DB_PORT = os.getenv('DB_PORT')
DB_NAME = os.getenv('DB_NAME')
DB_USER = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')

engine = create_engine(f'postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}')

cities_df.to_sql('gasprices', con=engine, if_exists='replace', index=False)


14